# 01. Threads

**Tier:** Deep. **Why you'd use this:** Several slow network waits should overlap instead of running one after another. Threads help with waiting; the GIL still serializes pure Python bytecode.

**After this lesson you can:** overlap waits with threads; run work in a `ThreadPoolExecutor`; explain what the GIL serializes.
About 30 minutes. **Uses:** [Context managers](../12-data-model/02-context-managers-lesson.ipynb)


## Quick review

1. What does `with` guarantee about a context manager?
2. Name one reason to prefer a pool over starting raw `Thread` objects by hand.

<details><summary>Answers</summary>

`__exit__` runs on the way out, even after an error. A pool caps workers and gives you futures for results.
</details>


## The idea

A thread runs Python callables while other threads can wait on I/O. `ThreadPoolExecutor` maps a function over items in a pool. The GIL (global interpreter lock) still allows only one thread to run Python bytecode at a time in the default build, so CPU-heavy Python rarely speeds up with threads. I/O waits release the GIL, so overlapping downloads or sleeps can finish sooner wall-clock wise.

Threads share one process address space. That makes sharing a list or counter easy and also makes races easy. The scheduler can switch threads between bytecode instructions. Waiting on I/O releases the GIL so another thread can run Python. Pure CPU loops mostly fight for the GIL instead of speeding up. Prefer a small number of worker threads for I/O; measure before adding more.

<details><summary>Explain more</summary>

Think of one microphone (the GIL) in a room of speakers. Only one talks Python at a time, but while one waits for the network, another can talk. Free-threaded builds change the microphone rules; locks still matter for shared data (next lessons).
</details>


## See it work

Square numbers in a small pool.


In [1]:
from concurrent.futures import ThreadPoolExecutor


def squares(nums):
    def square(value):
        # 1. Pure CPU here; pool still useful for structure.
        return value * value

    # 2. map keeps input order in the result list.
    with ThreadPoolExecutor(max_workers=2) as pool:
        return list(pool.map(square, nums))


print(squares([1, 2, 3, 4]))


[1, 4, 9, 16]


## Mental model

Think of threads as workers sharing one workshop. Tools on the bench (lists, counters, file handles) are visible to everyone. That is convenient when two waiters should append to the same results list. It is dangerous when two workers edit the same notebook line at once. The GIL is a door that only one Python speaker can pass at a time in the default build. Waiting on the network opens the door for others. Spinning on CPU keeps the door occupied.

| situation | threads help? | why |
|---|---|---|
| many HTTP GETs with `urllib` | yes | waits release the GIL |
| hashing a huge file in pure Python | usually no | CPU holds the GIL |
| calling a C extension that releases the GIL | sometimes | native code can run parallel |
| mutating a shared dict | only with a lock | races otherwise |

Join every non-daemon thread you start. Prefer `ThreadPoolExecutor` when the work is "map this function over items" so you do not hand-roll start/join. Pass data in via arguments; return data via futures or a queue. Avoid module-level mutable globals unless they are protected.


## Check yourself

Answer these from the example above before you run the cell.


1. Why can three `sleep` calls finish near one sleep's duration?
2. Why does `pool.map` return results in input order even if workers finish out of order?
3. What does the GIL not protect: your `list.append` invariants, or bytecode dispatch?

<details><summary>Answers</summary>

1. Waiting releases the GIL so workers overlap. 2. `map` gathers by submission order. 3. It serializes bytecode; your data races still need locks.
</details>


In [2]:
import time
from concurrent.futures import ThreadPoolExecutor


def nap(label):
    time.sleep(0.05)
    return label


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(nap, ["a", "b", "c"]))
elapsed = time.perf_counter() - start
print(results)
print(elapsed < 0.12)


['a', 'b', 'c']
True


<details><summary>Check your answers</summary>

Results are `['a', 'b', 'c']` in input order. Elapsed is under about 0.12 seconds because the three sleeps overlap, not 0.15. CPython detail: timings vary; the boolean should stay true on a quiet machine.
</details>


## Why it works this way

A thread is an OS-scheduled path of execution inside one process. Threads share the heap, so a list appended in one thread is visible in another without pickling. Sharing is also the hazard: two threads can interleave a read-modify-write and lose an update. The Global Interpreter Lock (GIL) in the default build allows only one thread to execute Python bytecode at a time. Waiting on I/O or sleeping releases the GIL, so other threads progress while one waits. CPU-bound loops rarely get wall-clock speedup from threads under the GIL; they take turns. Free-threaded builds remove that serialisation for bytecode, which makes races easier to hit and makes CPU scaling possible when the workload is parallelisable. Deep dive: [The GIL and free-threading](../../internals/12-gil/01-gil-lesson.ipynb).

Cost: each thread needs a stack (often about a megabyte reserved) and scheduler overhead. Creating thousands of threads for tiny sleeps is usually worse than a pool or async. Design reason for `threading.Thread` and `ThreadPoolExecutor`: reuse the blocking libraries you already have while overlapping waits. Prefer a queue or an explicit `Lock` when mutating shared state. Join threads you start so the main thread does not exit early. Daemon threads die with the process and should not own cleanup.

Trace of a two-thread I/O overlap: both start, both call a blocking wait, the GIL is released during the wait, both waits finish near the same wall time, then both write results. Evidence cells print overlapping sleeps finishing faster than sequential, and a pool mapping a list of URLs or labels in order of completion vs submission depending on API.


In [3]:
import time
from concurrent.futures import ThreadPoolExecutor


def nap(_):
    time.sleep(0.05)
    return 1


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    total = sum(pool.map(nap, range(3)))
print(total)
print(time.perf_counter() - start < 0.12)


3
True


## When to use it

Use threads for concurrent I/O-bound work in one process. Use processes for CPU-bound Python. Use async when many waits share one thread cooperatively. Don't expect threads to speed pure Python number crunching under the GIL. Don't share lists or counters across threads without a lock.

Choose threads when the library is blocking and you need overlap. Choose processes for CPU. Choose async when you control the I/O stack and want many waiters on one thread. Do not share mutable structures without a lock or a queue. Do not assume more threads always mean lower latency once the machine is saturated.


## Common mistake

Starting threads for heavy pure-Python CPU work and expecting a linear speedup.


In [4]:
import time
from concurrent.futures import ThreadPoolExecutor


def crunch(_):
    total = 0
    for value in range(200_000):
        total += value * value
    return total


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(crunch, range(4)))
threaded = time.perf_counter() - start
start = time.perf_counter()
for _ in range(4):
    crunch(0)
serial = time.perf_counter() - start
print(threaded < serial * 0.7)


False


Often the boolean is false or barely true: the GIL serializes the bytecode. Prefer processes for that shape of work.


In [5]:
print("Use ProcessPoolExecutor for CPU-bound Python; threads for waiting.")


Use ProcessPoolExecutor for CPU-bound Python; threads for waiting.


## Try it

Change `squares` to use `max_workers=1`. Does the result change, and what happens to overlapping naps?


## Recap

- Thread pools overlap waiting work in one process.
- The GIL limits CPU-bound Python threads.
- Protect shared data with locks; do not treat the GIL as your mutex.

**Next:** [practice](01-threads-practice.ipynb)
